# 🐾 Snack4Pets — French UGC Voice Generation

Generate high-converting French UGC voiceovers for **Snack4Pets** ads, TikToks, and Reels.

Supports two modes:
1. **Zero-Shot Voice Cloning (Zero training needed):** Clone any French pet-owner audio clip (5–10s) on the fly using the pre-trained base model.
2. **Custom Fitted Checkpoint:** Load your fine-tuned model from Google Drive for 100% voice consistency across all campaigns.

## 1. Setup GPU & Dependencies

In [ ]:
!nvidia-smi
!apt-get -y update && apt-get install -y ffmpeg
!pip install git+https://github.com/SWivid/F5-TTS.git
!pip install soundfile librosa gradio torchaudio

## 2. Mount Google Drive (To access checkpoints & reference audio)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

## 3. Load Model (Base or Fitted)

In [ ]:
import torch
from f5_tts.api import F5TTS
from IPython.display import Audio, display
import soundfile as sf

# Set to path of your custom .safetensors if using fitted voice, or None for Base Zero-Shot
CUSTOM_CHECKPOINT_PATH = None
# Example if fine-tuned: CUSTOM_CHECKPOINT_PATH = '/content/drive/MyDrive/snack4pets_tts_checkpoints/model_last.safetensors'

if CUSTOM_CHECKPOINT_PATH and os.path.exists(CUSTOM_CHECKPOINT_PATH):
    print(f"Loading fitted checkpoint: {CUSTOM_CHECKPOINT_PATH}")
    tts = F5TTS(model="F5TTS_v1_Base", ckpt_file=CUSTOM_CHECKPOINT_PATH)
else:
    print("Loading standard pre-trained F5-TTS Base model for zero-shot cloning...")
    tts = F5TTS(model="F5TTS_v1_Base")

print("Model ready for UGC generation.")

## 4. UGC Voice Generation Function
French UGC tip: Punctuation (commas, ellipses `...`, exclamation marks) directly impacts the speech rhythm and natural breath pauses.

In [ ]:
def generate_snack4pets_ugc(ref_audio_path, ref_text, gen_text, output_path="snack4pets_voiceover.wav", speed=1.1):
    """
    ref_audio_path: 5-10 second clean casual French voice sample
    ref_text: Exact transcript of that reference audio
    gen_text: The Snack4Pets UGC ad script to speak
    speed: Pacing multiplier (1.05 - 1.15 is ideal for energetic TikTok ads)
    """
    wav, sr, _ = tts.infer(
        ref_file=ref_audio_path,
        ref_text=ref_text,
        gen_text=gen_text,
        speed=speed
    )
    sf.write(output_path, wav, sr)
    print(f"Generated: {output_path} (Sample rate: {sr}Hz)")
    return output_path

## 5. Example Snack4Pets UGC Ad Script Generation

In [ ]:
# Clone repository reference voices if not present
import os
if not os.path.exists('/content/snack4pets-ugc-tts'):
    !git clone https://github.com/mohaidoss/snack4pets-ugc-tts.git /content/snack4pets-ugc-tts

# Default reference audio extracted from your real French pet UGC video
REF_AUDIO = '/content/snack4pets-ugc-tts/reference_voices/tao_chew_sample.wav'
REF_TEXT = "Tao a mis presque 16 minutes pour la finir. C'est une mastication qui dure entre 10 et 20 minutes en fonction des chiens. Elle fait partie de la catégorie moyenne durée."

# UGC Script for Snack4Pets Chews / Treats
AD_SCRIPT = "Si ton chien passe ses journées à s'ennuyer ou à ronger tes meubles, écoute bien ! Chez Snack4Pets, ils font des friandises 100% naturelles, sans additifs bizarres. Ça l'occupe pendant des heures, et ça nettoie ses dents naturellement. Franchement, le pack mastication a sauvé mon canapé. Teste, tu verras la différence tout de suite !"

out_file = generate_snack4pets_ugc(REF_AUDIO, REF_TEXT, AD_SCRIPT, 'ad_chews_v1.wav', speed=1.12)
display(Audio(out_file))


## 6. Launch Interactive Generation WebUI
If you want a live UI with audio playback, waveform preview, and sliders for speed and reference audio.

In [ ]:
!f5-tts_infer-gradio --share